### Motor Imagery Decoding with Riemannian Geometry and Physics-Based Simulation


## Introduction

This notebook documents a complete, end-to-end pipeline for decoding motor imagery intent from scalp EEG and translating classified commands into robot arm motion. Beginning from raw `.gdf` session files, the pipeline performs artefact rejection via ICA, SMR-band filtering, and Common Average Referencing before extracting fixed-length epochs time-locked to cue onset. Classification is performed in the Riemannian geometry framework: per-trial covariance matrices are estimated with Ledoit-Wolf shrinkage, projected onto the tangent space at the Riemannian mean, and classified with a linear SVM. Generalisation is evaluated under a Leave-One-Subject-Out protocol, with an adaptive few-shot calibration stage that augments the population model with a small number of subject-specific trials to improve transfer accuracy. The final stage drives a KUKA iiwa arm in a PyBullet physics simulation using majority-voted classifier outputs, with a QA visualisation stage providing per-subject diagnostics across all preprocessing and feature extraction steps.

## Dataset
This project uses the BCI Competition IV Dataset 2a, a widely adopted benchmark for motor imagery classification research. The dataset was recorded at the Institute for Knowledge Discovery, Graz University of Technology, and made publicly available through the BCI Competition IV (2008). It comprises 22-channel EEG recordings from nine subjects performing four-class motor imagery — left hand, right hand, both feet, and tongue — across two sessions per subject (training and evaluation). Full details of the acquisition protocol, electrode placement, and event structure are documented in:

Brunner, C., Leeb, R., Müller-Putz, G. R., Schlögl, A., & Pfurtscheller, G. (2008). *BCI Competition 2008 – Graz Data Set A*. Institute for Knowledge Discovery, Graz University of Technology, Austria.

## Imports & Environment Setup

In [9]:
import os
import time
import random
from collections import deque

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import mne
from mne.preprocessing import ICA

import pybullet as p
import pybullet_data

from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.model_selection import LeaveOneGroupOut, train_test_split
from sklearn.metrics import accuracy_score
from sklearn.decomposition import PCA

from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace
import imageio

## Continuous Preprocessing

The pipeline applies preprocessing to the continuous signal before any epoching, which is the correct order of operations: ICA decomposition requires a stable, sufficiently long signal to estimate independent components reliably, and filtering before epoching avoids transient edge artifacts that would otherwise appear at epoch boundaries.

A 1 Hz high-pass filter is applied prior to ICA rather than the final 8 Hz SMR bandpass because ICA is sensitive to slow drifts — low-frequency baseline wander inflates the covariance structure and causes variance to leak into components that should reflect neural or ocular sources. The SMR bandpass (8–30 Hz) is deferred until after ICA application for this reason. The firwin design is used throughout for its linear phase response, which preserves the temporal structure of the signal.

ICA is constrained to 15 components rather than the full channel rank. With 22 EEG channels, retaining all components risks fragmentation of artifact variance across many components, making EOG identification via cross-correlation less reliable. Fixing random_state ensures reproducibility across the subject-session loop. EOG detection omits the ch_name argument so that find_bads_eog cross-correlates candidate components against all three mapped EOG channels simultaneously — any component flagged by at least one channel is excluded. The deduplication step handles the case where multiple EOG channels flag the same component index, which would otherwise cause redundant entries in ica.exclude.
Common Average Reference is applied after the bandpass to avoid reintroducing low-frequency structure. CAR is appropriate here given the dense, homogeneous electrode layout of the BCI Competition IV 2a montage, where a global mean reference effectively attenuates spatially unstructured noise.

In [4]:
mne.set_log_level('ERROR')

input_dir = "raweegdata"
output_dir = "final_clean_eeg"
os.makedirs(output_dir, exist_ok=True)

subjects = [f"A0{i}" for i in range(1, 10)]
sessions = ["T", "E"]

for sub in subjects:
    for sess in sessions:
        filename = f"{sub}{sess}.gdf"
        filepath = os.path.join(input_dir, filename)
        
        if os.path.exists(filepath):
            raw = mne.io.read_raw_gdf(filepath, preload=True)
            raw.set_channel_types({'EOG-left': 'eog', 'EOG-central': 'eog', 'EOG-right': 'eog'})
            
            # High-pass at 1.0 Hz stabilizes baseline to prevent drift during ICA.
            raw.filter(l_freq=1.0, h_freq=None, fir_design='firwin')
            
            # Restricting ICA to 15 components consolidates artifacts and prevents variance fragmentation.
            ica = ICA(n_components=15, max_iter='auto', random_state=97)
            ica.fit(raw)
            
            # Omitting ch_name forces cross-correlation against all mapped EOG channels.
            eog_indices, eog_scores = ica.find_bads_eog(raw)
            
            if eog_indices:
                # Deduplicate indices to handle overlapping flags from multiple EOG channels.
                unique_eog_indices = list(set(eog_indices))
                ica.exclude = unique_eog_indices
                
            ica.apply(raw)
            
            # Applying SMR bandpass (8-30 Hz) on continuous data prevents epoching edge artifacts.
            raw.filter(l_freq=8.0, h_freq=30.0, fir_design='firwin')
            
            # CAR improves spatial resolution by subtracting global noise.
            raw.set_eeg_reference(ref_channels='average')
            
            out_filename = f"{sub}{sess}_clean.fif"
            out_filepath = os.path.join(output_dir, out_filename)
            raw.save(out_filepath, overwrite=True)
            
            del raw, ica

## Epoching & Artifact Scrubbing

Epoching is performed on the preprocessed continuous data, extracting fixed-length segments time-locked to motor imagery cue onset. The epoch window spans 0.5 s to 3.5 s relative to the cue, deliberately excluding the first 500 ms to avoid the visual evoked potential elicited by the cue stimulus itself, which would contaminate the SMR features with a non-motor response. No baseline correction is applied because the upstream bandpass (8–30 Hz) has already removed DC offset and slow drift, making baseline subtraction redundant and potentially distorting the early SMR window.

The BCI Competition IV 2a dataset annotates expert-identified artifact trials with event code `1023`. Rather than relying solely on these flags, the pipeline enforces a ±1000-sample exclusion zone around each flagged sample, dropping any motor imagery trial whose onset falls within that window. This guards against trials that are temporally adjacent to a flagged artifact but not themselves marked — a common source of label noise in competitive EEG datasets where artifact boundaries are not always sharp.

Event code handling requires a dynamic mapping because evaluation session files (suffix `E`) encode cue onsets as `783` (unknown class) rather than the labelled codes `769–772` used in training sessions. The mapping is constructed at runtime from the intersection of expected codes and those actually present in each file, preventing MNE from raising a `ValueError` on absent keys. The `783` branch assigns a placeholder label of 0, preserving trial structure for evaluation files where true labels are withheld.

In [5]:
mne.set_log_level('ERROR')

input_dir = "final_clean_eeg"
output_dir = "epoched_eeg_data"
os.makedirs(output_dir, exist_ok=True)

clean_files = [f for f in os.listdir(input_dir) if f.endswith('_clean.fif')]

if not clean_files:
    print(f"No '_clean.fif' files found in '{input_dir}'.")
else:
    motor_imagery_mapping = {
        '769': 1,  # Left Hand
        '770': 2,  # Right Hand
        '771': 3,  # Both Feet
        '772': 4   # Tongue
    }

    for filename in clean_files:
        filepath = os.path.join(input_dir, filename)
        print(f"\nProcessing {filename}...")

        raw = mne.io.read_raw_fif(filepath, preload=True)
        all_events, all_event_dict = mne.events_from_annotations(raw)

        if '1023' in all_event_dict:
            artifact_id = all_event_dict['1023']
            artifact_samples = all_events[all_events[:, 2] == artifact_id, 0]
            print(f"  {len(artifact_samples)} expert-flagged artifact markers found.")
        else:
            artifact_samples = []

        if '783' in all_event_dict and not any(k in all_event_dict for k in ['769', '770', '771', '772']):
            dynamic_mapping = {'783': 0}
        else:
            # Only map event codes present in this file — missing codes cause MNE to crash
            dynamic_mapping = {k: v for k, v in motor_imagery_mapping.items() if k in all_event_dict}

        if not dynamic_mapping:
            print(f"  No motor imagery cues found in {filename}, skipping.")
            del raw
            continue

        mi_events, mi_event_dict = mne.events_from_annotations(raw, event_id=dynamic_mapping)

        # Drop trials within ±1000 samples of an expert artifact flag
        clean_mi_events = [
            e for e in mi_events
            if not any(abs(a - e[0]) <= 1000 for a in artifact_samples)
        ]
        clean_mi_events = np.array(clean_mi_events)
        print(f"  {len(mi_events) - len(clean_mi_events)} trials dropped. {len(clean_mi_events)} remaining.")

        if len(clean_mi_events) == 0:
            print(f"  No clean trials remaining in {filename}, skipping.")
            del raw
            continue

        epochs = mne.Epochs(
            raw,
            clean_mi_events,
            event_id=mi_event_dict,
            tmin=0.5,
            tmax=3.5,
            baseline=None,
            preload=True
        )

        out_filename = filename.replace('_clean.fif', '_clean-epo.fif')
        out_filepath = os.path.join(output_dir, out_filename)
        epochs.save(out_filepath, overwrite=True)
        print(f"  Saved: {out_filename}")

        del raw, epochs


Processing A01E_clean.fif...
  7 expert-flagged artifact markers found.
  7 trials dropped. 281 remaining.
  Saved: A01E_clean-epo.fif

Processing A01T_clean.fif...
  15 expert-flagged artifact markers found.
  15 trials dropped. 273 remaining.
  Saved: A01T_clean-epo.fif

Processing A02E_clean.fif...
  5 expert-flagged artifact markers found.
  5 trials dropped. 283 remaining.
  Saved: A02E_clean-epo.fif

Processing A02T_clean.fif...
  18 expert-flagged artifact markers found.
  18 trials dropped. 270 remaining.
  Saved: A02T_clean-epo.fif

Processing A03E_clean.fif...
  15 expert-flagged artifact markers found.
  15 trials dropped. 273 remaining.
  Saved: A03E_clean-epo.fif

Processing A03T_clean.fif...
  18 expert-flagged artifact markers found.
  18 trials dropped. 270 remaining.
  Saved: A03T_clean-epo.fif

Processing A04E_clean.fif...
  60 expert-flagged artifact markers found.
  60 trials dropped. 228 remaining.
  Saved: A04E_clean-epo.fif

Processing A04T_clean.fif...
  26 exp

## Multi-Subject Data Loading

This stage consolidates the preprocessed, epoched data from all subjects into three aligned arrays — `X_all`, `y_all`, and `groups_all` — structured for scikit-learn's group-aware cross-validation interface. The `groups_all` array encodes subject identity as an integer, which downstream `LeaveOneGroupOut` splitting uses to ensure no subject's data appears in both train and test folds simultaneously.

Evaluation session files are explicitly excluded by detecting single-class label arrays: because the `E` session files have their true labels withheld (all onsets encoded as `783` → 0), their epoch arrays carry only one unique class value, making them uninformative for supervised training and incompatible with stratified metrics. Only files containing at least two distinct class labels are admitted to the concatenated arrays.

The crop to 0.5–3.5 s is applied again here despite having been set as the epoch window in Stage 2. This is a defensive measure: MNE's `.save`/`.read_epochs` cycle can occasionally widen the stored time axis by a sample or two depending on floating-point boundary handling, and re-cropping guarantees a consistent, identical time axis across all subjects before the data enters the covariance estimator in Stage 4. Picking `'eeg'` during `get_data` drops the three EOG channels that remain in the `Epochs` object, ensuring the feature tensor contains only the 22 EEG channels used for Riemannian covariance estimation.

In [6]:

mne.set_log_level('ERROR')

data_dir = 'epoched_eeg_data'
X_all = []
y_all = []
groups_all = []

try:
    epoch_files = sorted([f for f in os.listdir(data_dir) if f.endswith('_clean-epo.fif')])
except FileNotFoundError:
    raise FileNotFoundError(f"Directory '{data_dir}' not found. Ensure previous cells ran successfully.")

if not epoch_files:
    raise ValueError(f"No '_clean-epo.fif' files found in '{data_dir}'.")

for filename in epoch_files:
    try:
        subject_id = int(filename[1:3])
    except ValueError:
        print(f"  Could not parse integer subject ID in {filename}, skipping.")
        continue

    filepath = os.path.join(data_dir, filename)
    epochs = mne.read_epochs(filepath, preload=True)
    
    # Crop to isolate Event-Related Desynchronization
    epochs.crop(tmin=0.5, tmax=3.5)
    
    y_temp = epochs.events[:, 2]
    unique_classes = np.unique(y_temp)
    
    if len(unique_classes) > 1:
        # Pick 'eeg' to automatically drop EOG channels
        X_temp = epochs.get_data(picks='eeg')
        
        group_temp = np.full(len(y_temp), subject_id)
        
        X_all.append(X_temp)
        y_all.append(y_temp)
        groups_all.append(group_temp)
        
    else:
         print(f"  Skipping evaluation data: {filename}")
         
    del epochs

if X_all:
    X_all = np.concatenate(X_all, axis=0)
    y_all = np.concatenate(y_all, axis=0)
    groups_all = np.concatenate(groups_all, axis=0)
    
    print(f"Final Feature Tensor (X_all) Shape: {X_all.shape}")
    print(f"Final Labels Array (y_all) Shape: {y_all.shape}")
    print(f"Final Groups Array (groups_all) Shape: {groups_all.shape}")
    print(f"Subjects Loaded into Global Array: {np.unique(groups_all)}")
    
else:
    print("No valid training data found to concatenate.")

  Skipping evaluation data: A01E_clean-epo.fif
  Skipping evaluation data: A02E_clean-epo.fif
  Skipping evaluation data: A03E_clean-epo.fif
  Skipping evaluation data: A04E_clean-epo.fif
  Skipping evaluation data: A05E_clean-epo.fif
  Skipping evaluation data: A06E_clean-epo.fif
  Skipping evaluation data: A07E_clean-epo.fif
  Skipping evaluation data: A08E_clean-epo.fif
  Skipping evaluation data: A09E_clean-epo.fif
Final Feature Tensor (X_all) Shape: (2328, 22, 751)
Final Labels Array (y_all) Shape: (2328,)
Final Groups Array (groups_all) Shape: (2328,)
Subjects Loaded into Global Array: [1 2 3 4 5 6 7 8 9]


## Riemannian Geometry & SVM Classification (Generalised LOSO)

The classification pipeline operates on the symmetric positive definite (SPD) manifold rather than in the raw electrode-time feature space. Each epoch's 22-channel EEG segment is summarised as a 22×22 covariance matrix, and the Riemannian metric is used to map these matrices to a Euclidean tangent space at the Riemannian mean of the training set. This projection, implemented via `TangentSpace(metric='riemann')`, yields a vectorised feature representation that respects the geometry of the SPD manifold — an approach shown by Barachant et al. (2012) to substantially outperform Euclidean covariance features for motor imagery classification.

Covariance estimation uses Ledoit-Wolf shrinkage (`estimator='lwf'`) rather than the sample covariance. With epoch lengths of 3 seconds at typical EEG sampling rates, the number of time samples per epoch is not large relative to the 22-channel dimensionality, placing estimation in a regime where the sample covariance is poorly conditioned. Ledoit-Wolf shrinkage applies an analytically optimal regularisation coefficient (Ledoit & Wolf 2004) that pulls the estimate toward a scaled identity, improving both conditioning and generalisation of the downstream tangent space projection.

The evaluation protocol is Leave-One-Subject-Out (LOSO) cross-validation, where each fold trains on all subjects except one and tests on the held-out subject. This directly measures transfer accuracy — the ability of a model trained on the population to generalise to an unseen individual — which is the operationally relevant metric for a generalised BCI that requires no per-subject calibration. A linear SVM with `C=1.0` is used as the final classifier; given that tangent space projection already linearises the class boundaries on the SPD manifold, a linear kernel is both theoretically motivated and computationally efficient at this feature dimensionality.

In [7]:


# Ledoit-Wolf shrinkage stabilizes multi-subject covariance matrices
clf_pipeline = Pipeline([
    ('Covariances', Covariances(estimator='lwf')),
    ('TangentSpace', TangentSpace(metric='riemann')),
    ('SVM', SVC(kernel='linear', C=1.0))
])

logo = LeaveOneGroupOut()

fold_accuracies = []
subjects_tested = []

for fold_idx, (train_idx, test_idx) in enumerate(logo.split(X_all, y_all, groups_all)):
    X_train, y_train = X_all[train_idx], y_all[train_idx]
    X_test, y_test = X_all[test_idx], y_all[test_idx]
    
    test_subject_id = np.unique(groups_all[test_idx])[0]
    
    clf_pipeline.fit(X_train, y_train)
    y_pred = clf_pipeline.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    fold_accuracies.append(acc)
    subjects_tested.append(test_subject_id)
    
    print(f"  Fold {fold_idx + 1} | Target Subject: A0{test_subject_id} | Accuracy: {acc * 100:.2f}%")

print(f"Mean Transfer Accuracy: {np.mean(fold_accuracies) * 100:.2f}% +/- {np.std(fold_accuracies) * 100:.2f}%")

  Fold 1 | Target Subject: A01 | Accuracy: 45.42%
  Fold 2 | Target Subject: A02 | Accuracy: 25.56%
  Fold 3 | Target Subject: A03 | Accuracy: 56.30%
  Fold 4 | Target Subject: A04 | Accuracy: 31.68%
  Fold 5 | Target Subject: A05 | Accuracy: 24.05%
  Fold 6 | Target Subject: A06 | Accuracy: 29.22%
  Fold 7 | Target Subject: A07 | Accuracy: 26.94%
  Fold 8 | Target Subject: A08 | Accuracy: 42.80%
  Fold 9 | Target Subject: A09 | Accuracy: 43.88%
Mean Transfer Accuracy: 36.20% +/- 10.58%


## Dynamic Calibration (Offline)

Where the previous cell evaluated a purely generalised model with no subject-specific information, this stage introduces few-shot calibration to adapt the population-level classifier to the held-out subject. A small number of labelled trials from the test subject are pooled with the full cross-subject training set before refitting, exploiting the fact that even a handful of subject-specific covariance matrices can shift the tangent space projection toward the individual's signal geometry without discarding the population prior encoded in the larger training set.

The calibration budget is determined adaptively via a `while` loop that begins at 3 shots per class (12 trials total across 4 classes) and increments by 2 until either a 65% accuracy threshold is met or the per-class shot count reaches 15. The 65% target is chosen as a conservative operating threshold above chance (25% for 4-class MI) that justifies robot arm actuation; the hard cap at 15 shots per class prevents calibration from consuming the majority of the test subject's available trials, which would produce an overly optimistic accuracy estimate on the residual evaluation set.

Calibration trials are drawn from the test subject's data using stratified splitting with a fixed `random_state`, ensuring balanced class representation across all shot counts and reproducible fold boundaries. The remainder after calibration extraction forms the evaluation set — meaning accuracy is always measured on trials the model has not seen, preserving the integrity of the LOSO estimate. The pipeline object is refit from scratch on each calibration attempt rather than fine-tuned, which is correct here because `TangentSpace` must recompute the Riemannian mean of the augmented training set; incremental updates to the mean are not supported by pyriemann's current tangent space implementation.

In [8]:

# Ledoit-Wolf shrinkage stabilizes multi-subject covariance matrices
clf_pipeline = Pipeline([
    ('Covariances', Covariances(estimator='lwf')), 
    ('TangentSpace', TangentSpace(metric='riemann')),
    ('SVM', SVC(kernel='linear', C=1.0))
])

logo = LeaveOneGroupOut()
fold_accuracies = []

for fold_idx, (train_idx, test_idx) in enumerate(logo.split(X_all, y_all, groups_all)):
    X_train_base, y_train_base = X_all[train_idx], y_all[train_idx]
    X_test_full, y_test_full = X_all[test_idx], y_all[test_idx]
    
    test_subject_id = np.unique(groups_all[test_idx])[0]
    
    shots_per_class = 3
    # Hard limit prevents calibration from consuming the entire test set
    max_shots = 15      
    acc = 0.0
    
    while acc < 0.65 and shots_per_class <= max_shots:
        total_calib_trials = shots_per_class * 4
        
        if total_calib_trials >= len(y_test_full):
            print(f"  Hit maximum available trials for A0{test_subject_id}, breaking.")
            break
        
        # Stratification ensures balanced class distribution during few-shot calibration
        # Fixed random state ensures reproducible cross-validation splits
        X_eval, X_calib, y_eval, y_calib = train_test_split(
            X_test_full, y_test_full, 
            test_size=total_calib_trials, 
            stratify=y_test_full,
            random_state=42
        )
        
        X_train_calibrated = np.concatenate((X_train_base, X_calib), axis=0)
        y_train_calibrated = np.concatenate((y_train_base, y_calib), axis=0)
        
        clf_pipeline.fit(X_train_calibrated, y_train_calibrated)
        y_pred = clf_pipeline.predict(X_eval)
        
        acc = accuracy_score(y_eval, y_pred)
        
        print(f"  {shots_per_class}-shot attempt: {acc * 100:.2f}%")
        
        if acc < 0.65:
            shots_per_class += 2 
            
    fold_accuracies.append(acc)
    print(f"  Fold complete, final accuracy: {acc * 100:.2f}%")

print(f"Mean Transfer Accuracy: {np.mean(fold_accuracies) * 100:.2f}% +/- {np.std(fold_accuracies) * 100:.2f}%")

  3-shot attempt: 52.11%
  5-shot attempt: 55.34%
  7-shot attempt: 57.96%
  9-shot attempt: 62.45%
  11-shot attempt: 62.01%
  13-shot attempt: 63.35%
  15-shot attempt: 63.85%
  Fold complete, final accuracy: 63.85%
  3-shot attempt: 26.74%
  5-shot attempt: 31.60%
  7-shot attempt: 30.58%
  9-shot attempt: 32.48%
  11-shot attempt: 33.19%
  13-shot attempt: 33.94%
  15-shot attempt: 33.33%
  Fold complete, final accuracy: 33.33%
  3-shot attempt: 66.28%
  Fold complete, final accuracy: 66.28%
  3-shot attempt: 42.40%
  5-shot attempt: 42.15%
  7-shot attempt: 42.31%
  9-shot attempt: 42.92%
  11-shot attempt: 44.95%
  13-shot attempt: 44.29%
  15-shot attempt: 42.57%
  Fold complete, final accuracy: 42.57%
  3-shot attempt: 30.80%
  5-shot attempt: 32.64%
  7-shot attempt: 35.90%
  9-shot attempt: 34.07%
  11-shot attempt: 36.70%
  13-shot attempt: 34.29%
  15-shot attempt: 37.13%
  Fold complete, final accuracy: 37.13%
  3-shot attempt: 35.75%
  5-shot attempt: 39.20%
  7-shot atte

KeyboardInterrupt: 

## Robot Arm Simulation

This cell closes the BCI loop by feeding classifier outputs into a physics simulation, providing a behavioural substrate against which the pipeline's temporal stability and command fidelity can be assessed. The simulation uses a KUKA iiwa 7-DOF arm loaded into PyBullet under standard gravity, chosen because its URDF is bundled with pybullet_data and its joint structure is sufficiently articulated to produce visually distinct responses to different command classes. The intent is not to replicate a clinically deployable neuroprosthetic controller — joint mapping, inverse kinematics, and task-space planning are all absent — but to make classifier output observable as physically coherent motion, which is the minimum requirement for methodology substantiation.

The evaluation data passed into the simulation loop is sorted by class label before inference. This is a deliberate experimental choice rather than an attempt to replicate natural BCI operation: by grouping trials into sustained blocks of the same true intent, the sequence emulates the kind of continuous, held motor imagery that a user performing a real BCI task would produce. It also makes the majority-vote buffer's behaviour interpretable — within a class-homogeneous block, the buffer should converge rapidly to the correct command, and transitions between blocks expose the filter's lag. Without this sorting, the random interleaving of classes across the evaluation set would produce command sequences too volatile to assess visually.

Single-trial classifier outputs are passed through a rolling majority-vote buffer of width 5 before any joint command is issued. This is a standard temporal smoothing strategy in online BCI systems, motivated by the fact that EEG-based classifiers operating at the single-trial level carry substantial trial-to-trial variance even when mean accuracy is high. Requiring a buffer to fill before actuation begins (the `continue` guard for `len < buffer_size`) ensures the first command is always based on a full consensus window rather than a partial one, preventing spurious early motion.

The four motor imagery classes are mapped to joint velocity commands as follows: class 1 (left hand) drives joint 0 at +0.8 rad/s, producing a positive-direction rotation about the arm's base; class 2 (right hand) drives joint 0 at −0.8 rad/s, reversing that rotation; class 3 (both feet) drives joint 3 at +0.8 rad/s, flexing the arm at the elbow-equivalent joint; class 4 (tongue) simultaneously drives joint 0 at −0.8 rad/s and joint 3 at −0.8 rad/s, producing a compound retraction movement. It should be noted that classes 2 and 4 share joint 0 at the same velocity magnitude and direction — this is a limitation of the current mapping rather than an intentional design, and means the simulation does not produce a unique distinguishable motion for every class. A fully articulated mapping would assign each class a kinematically distinct configuration. Each command is sustained for 30 physics steps at 240 Hz (approximately 125 ms), which is consistent with the update rates used in real-time BCI decoders operating on non-overlapping 125–500 ms windows. The match/miss logging at every fifth block provides a lightweight ground-truth audit trail against `y_sorted` without printing redundantly on every step.

In [11]:
try:
    p.disconnect() 
except:
    pass

physicsClient = p.connect(p.GUI)
p.setAdditionalSearchPath(pybullet_data.getDataPath())
p.setGravity(0, 0, -9.81)

plane_id = p.loadURDF("plane.urdf")
arm_id = p.loadURDF("kuka_iiwa/model.urdf", [0, 0, 0], useFixedBase=True)

p.resetDebugVisualizerCamera(cameraDistance=1.5, cameraYaw=45, cameraPitch=-30, cameraTargetPosition=[0, 0, 0.5])
p.configureDebugVisualizer(p.COV_ENABLE_GUI, 0)

video_frames = []

# Sorting by class simulates continuous, sustained human intent for realistic BCI testing.
sorted_indices = np.argsort(y_eval)
X_sorted = X_eval[sorted_indices]
y_sorted = y_eval[sorted_indices]

buffer_size = 5
prediction_buffer = deque(maxlen=buffer_size)

for i in range(len(X_sorted)):
    X_live = np.expand_dims(X_sorted[i], axis=0)
    true_intent = y_sorted[i]
    
    raw_prediction = clf_pipeline.predict(X_live)[0]
    prediction_buffer.append(raw_prediction)
    
    if len(prediction_buffer) < buffer_size:
        continue
        
    # A rolling majority-vote buffer prevents isolated misclassifications from causing erratic robot movements.
    majority_vote = max(set(prediction_buffer), key=prediction_buffer.count)
    
    if majority_vote == 1:
        p.setJointMotorControl2(arm_id, jointIndex=0, controlMode=p.VELOCITY_CONTROL, targetVelocity=0.8)
    elif majority_vote == 2:
        p.setJointMotorControl2(arm_id, jointIndex=0, controlMode=p.VELOCITY_CONTROL, targetVelocity=-0.8)
    elif majority_vote == 3:
        p.setJointMotorControl2(arm_id, jointIndex=3, controlMode=p.VELOCITY_CONTROL, targetVelocity=0.8)
    elif majority_vote == 4:
        p.setJointMotorControl2(arm_id, jointIndex=0, controlMode=p.VELOCITY_CONTROL, targetVelocity=-0.8)
        p.setJointMotorControl2(arm_id, jointIndex=3, controlMode=p.VELOCITY_CONTROL, targetVelocity=-0.8)
    
    match = "Match" if majority_vote == true_intent else "Miss"
    
    if i % 5 == 0:
        print(f"  Block {i:03d} | True Intent: {true_intent} | Filtered Command: {majority_vote} [{match}]")

    # --- CHANGED: Renamed '_' to 'step' so we can count it ---
    for step in range(30):
        p.stepSimulation()
        time.sleep(1./240.)
        
        # --- NEW: Grab a camera frame every 4th step ---
        if step % 4 == 0:
            width, height, rgbImg, depthImg, segImg = p.getCameraImage(
                width=640, height=480, 
                renderer=p.ER_BULLET_HARDWARE_OPENGL
            )
            video_frames.append(rgbImg[:, :, :3])

print("Simulation Complete. Stitching video...")
time.sleep(2)

# --- CHANGED: Replaced stopStateLogging with imageio video writer ---
import imageio # Added here just in case it isn't in your imports cell
imageio.mimsave('clean_simulation.mp4', video_frames, fps=60)
print("Video saved successfully!")

p.disconnect()

  Block 005 | True Intent: 1 | Filtered Command: 1 [Match]
  Block 010 | True Intent: 1 | Filtered Command: 1 [Match]
  Block 015 | True Intent: 1 | Filtered Command: 1 [Match]
  Block 020 | True Intent: 1 | Filtered Command: 1 [Match]
  Block 025 | True Intent: 1 | Filtered Command: 1 [Match]
  Block 030 | True Intent: 1 | Filtered Command: 1 [Match]
  Block 035 | True Intent: 1 | Filtered Command: 1 [Match]
  Block 040 | True Intent: 1 | Filtered Command: 1 [Match]
  Block 045 | True Intent: 1 | Filtered Command: 1 [Match]
  Block 050 | True Intent: 1 | Filtered Command: 3 [Miss]
  Block 055 | True Intent: 1 | Filtered Command: 1 [Match]
  Block 060 | True Intent: 1 | Filtered Command: 3 [Miss]
  Block 065 | True Intent: 2 | Filtered Command: 2 [Match]
  Block 070 | True Intent: 2 | Filtered Command: 2 [Match]
  Block 075 | True Intent: 2 | Filtered Command: 2 [Match]
  Block 080 | True Intent: 2 | Filtered Command: 2 [Match]
  Block 085 | True Intent: 2 | Filtered Command: 2 [Match]

## Pipeline QA Visualisation

This cell produces a five-panel diagnostic figure for a randomly selected training subject, reconstructing intermediate signal states at each pipeline stage from saved artefacts rather than reprocessing from scratch. The subject is drawn at random to discourage overfitting the visual inspection to a single known-good case. Each panel targets a distinct transformation checkpoint, allowing the entire preprocessing and feature extraction chain to be audited from a single figure.

The first panel ( Raw EEG) plots 10 seconds of unprocessed continuous data across the first 10 channels, offset vertically by 20 µV per channel for legibility. The primary diagnostic signal here is the presence of large-amplitude, broadband transients coinciding across the three EOG channels and contaminating adjacent EEG channels — these are the blink and saccade artefacts that ICA is tasked with removing. A well-formed raw trace should show clearly visible spike-and-decay morphology in the EOG-proximal channels, confirming that the source data contains the artefact structure ICA needs to model.

The second panel (FIR Filtered) shows the same time window after applying the 1 Hz high-pass and 8–30 Hz SMR bandpass sequentially to the raw object, without ICA cleaning. Comparing this directly against the raw panel isolates the effect of spectral shaping alone. The expected observation is a marked reduction in slow drift and high-frequency muscle noise, with the signal amplitude substantially compressed relative to the raw trace, but EOG transients should remain partially visible because bandpass filtering attenuates but does not eliminate ocular artefacts within the SMR band. If the EOG spikes have fully disappeared at this stage, it suggests either that the artefacts were predominantly low-frequency (correctly removed by the high-pass) or that the particular 10-second window was artefact-free — both interpretations carry implications for how much work ICA is actually doing.

The third panel (ICA Cleaned) loads the saved post-ICA continuous file and plots the same time window. The critical comparison is against the second panel rather than the first: the bandpass filter has already been applied in both cases, so any residual difference is attributable solely to ICA component subtraction. A successful ICA cleaning should show a smoother, more spatially consistent set of traces with the sharp transient deflections characteristic of EOG propagation visibly attenuated. Absence of visible change between panels two and three would indicate either that no EOG components were flagged for this subject (check the Stage 1 console output) or that the flagged components did not carry appreciable variance in the SMR band.

The fourth panel (Mean Covariance Matrix) displays the Euclidean mean of the Ledoit-Wolf shrinkage covariance matrices computed across all trials for the selected subject, rendered as a 22×22 heatmap. The Euclidean mean is used here strictly for visualisation — the classifier operates on the Riemannian mean in tangent space — but it provides an interpretable summary of the channel covariance structure entering the classifier. A well-conditioned covariance matrix should show strong diagonal dominance, with the highest variance concentrated on channels overlying sensorimotor cortex (C3, Cz, C4 and their neighbours), and off-diagonal structure reflecting genuine spatial correlation rather than volume conduction artefacts. Symmetric off-diagonal clusters of high covariance between anatomically adjacent channels are expected and desirable; diffuse, uniformly high off-diagonal values would suggest residual common-mode noise or insufficient ICA cleaning.

The fifth panel (Tangent Space PCA) projects the full set of tangent space feature vectors — each a vectorised upper triangle of a projected SPD matrix — into two dimensions via PCA, and plots trials coloured by true class label. This scatter is the most direct visualisation of the classifier's operating conditions: the degree to which the four motor imagery classes form separable clusters in this 2D projection is a lower-bound indicator of linear separability in the full tangent space, since PCA discards all but the two highest-variance directions. Well-separated clusters with minimal overlap indicate that the Riemannian geometry pipeline is successfully extracting class-discriminative structure from the covariance matrices. Substantial overlap — particularly between left hand and right hand, which are the most spatially similar in terms of lateralised SMR suppression — is expected in a 2D projection and does not necessarily indicate poor classifier performance, as the linear SVM operates in the full tangent space dimensionality of 253 features (the upper triangle of a 22×22 matrix). The explained variance reported on each axis quantifies how much of the total tangent space variance is captured by the projection; low values (below 20% combined) indicate that the 2D view is a poor summary of the true feature geometry and visual cluster separation should be interpreted cautiously.

In [ ]:

mne.set_log_level('ERROR')

epoched_dir = "epoched_eeg_data"
epoched_files = [f for f in os.listdir(epoched_dir) if f.endswith('_clean-epo.fif') and 'T' in f]

if not epoched_files:
    print(f"No training epoch files found in {epoched_dir}, skipping.")
else:
    target_epo_file = random.choice(epoched_files)
    base_name = target_epo_file.split('_')[0]

    raw_filepath   = os.path.join("raweegdata", f"{base_name}.gdf")
    clean_filepath = os.path.join("final_clean_eeg", f"{base_name}_clean.fif")
    epo_filepath   = os.path.join(epoched_dir, target_epo_file)

    stage1_raw = mne.io.read_raw_gdf(raw_filepath, preload=True)
    stage1_raw.set_channel_types({'EOG-left': 'eog', 'EOG-central': 'eog', 'EOG-right': 'eog'})

    stage2_filtered = stage1_raw.copy()
    stage2_filtered.filter(l_freq=1.0, h_freq=None, fir_design='firwin')
    stage2_filtered.filter(l_freq=8.0, h_freq=30.0, fir_design='firwin')

    stage3_clean   = mne.io.read_raw_fif(clean_filepath, preload=True)
    stage4_epochs  = mne.read_epochs(epo_filepath, preload=True)

    X = stage4_epochs.get_data(picks='eeg')
    y = stage4_epochs.events[:, 2]

    # Use identical Ledoit-Wolf shrinkage as classifier to ensure visualizations reflect actual model inputs
    cov_estimator = Covariances(estimator='lwf')
    covs = cov_estimator.fit_transform(X)          

    # Euclidean mean used strictly for visual heatmap baseline, not for Riemannian calculations
    mean_cov = np.mean(covs, axis=0)

    # PCA reduction maps high-dimensional Riemannian geometry to 2D Euclidean space for visual scatter
    ts = TangentSpace(metric='riemann')
    X_tangent = ts.fit_transform(covs)             
    pca = PCA(n_components=2)
    X_2d = pca.fit_transform(X_tangent)            

    fig = plt.figure(figsize=(18, 22))
    fig.suptitle(f"Pipeline QA — Subject {base_name}", fontsize=16, fontweight='bold')

    gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35)

    ax_raw      = fig.add_subplot(gs[0, :])   
    ax_filtered = fig.add_subplot(gs[1, 0])   
    ax_ica      = fig.add_subplot(gs[1, 1])   
    ax_cov      = fig.add_subplot(gs[2, 0])   
    ax_scatter  = fig.add_subplot(gs[2, 1])   

    tmin_plot, tmax_plot = 5.0, 15.0

    def plot_raw_on_axes(raw_obj, ax, title, start, duration):
        data, times = raw_obj.get_data(
            start=raw_obj.time_as_index(start)[0],
            stop=raw_obj.time_as_index(start + duration)[0],
            return_times=True
        )
        for i in range(min(10, data.shape[0])):
            ax.plot(times, data[i, :] + (i * 20e-6), color='k', linewidth=0.5)
        ax.set_title(title, fontsize=11)
        ax.set_yticks([])
        ax.set_xlabel("Time (s)")

    plot_raw_on_axes(stage1_raw, ax_raw,
                     "Stage 1 — Raw EEG (10 channels, EOG spikes visible)",
                     tmin_plot, tmax_plot - tmin_plot)

    plot_raw_on_axes(stage2_filtered, ax_filtered,
                     "Stage 2 — FIR Filtered (1Hz HP + 8–30Hz BP)",
                     tmin_plot, tmax_plot - tmin_plot)

    plot_raw_on_axes(stage3_clean, ax_ica,
                     "Stage 3 — ICA Cleaned (EOG artifacts removed)",
                     tmin_plot, tmax_plot - tmin_plot)

    ch_names = stage4_epochs.copy().pick('eeg').ch_names
    im = ax_cov.imshow(mean_cov, cmap='RdBu_r', aspect='auto')
    ax_cov.set_title("Stage 4 — Mean Covariance Matrix (22×22 SPD)", fontsize=11)
    ax_cov.set_xlabel("Channel")
    ax_cov.set_ylabel("Channel")
    tick_step = 4
    ax_cov.set_xticks(range(0, len(ch_names), tick_step))
    ax_cov.set_xticklabels(ch_names[::tick_step], rotation=45, ha='right', fontsize=7)
    ax_cov.set_yticks(range(0, len(ch_names), tick_step))
    ax_cov.set_yticklabels(ch_names[::tick_step], fontsize=7)
    plt.colorbar(im, ax=ax_cov, fraction=0.046, pad=0.04)

    class_labels = {1: 'Left Hand', 2: 'Right Hand', 3: 'Feet', 4: 'Tongue'}
    colors       = {1: '#2196F3', 2: '#F44336', 3: '#4CAF50', 4: '#FF9800'}

    for cls in np.unique(y):
        mask = y == cls
        ax_scatter.scatter(
            X_2d[mask, 0], X_2d[mask, 1],
            label=class_labels.get(cls, str(cls)),
            color=colors.get(cls, 'gray'),
            alpha=0.6, s=40, edgecolors='none'
        )

    ax_scatter.set_title(
        "Stage 4 — Tangent Space (PCA 2D)\nClass separability in Riemannian feature space",
        fontsize=11
    )
    ax_scatter.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
    ax_scatter.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
    ax_scatter.legend(loc='best', fontsize=8)
    ax_scatter.axhline(0, color='gray', linewidth=0.5, linestyle='--')
    ax_scatter.axvline(0, color='gray', linewidth=0.5, linestyle='--')

    plt.savefig(f"pipeline_qa_{base_name}.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  Saved: pipeline_qa_{base_name}.png")

    del stage1_raw, stage2_filtered, stage3_clean, stage4_epochs